In [2]:
%pip install requests pandas pyarrow python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests
import pandas as pd
from io import StringIO
from pathlib import Path
import getpass

BASE_URL = "https://airquality.xkdr.org"
API_KEY = getpass.getpass("Enter your XKDR API key: ")

headers = {"Authorization": f"Bearer {API_KEY}"}

Path("../data/raw").mkdir(parents=True, exist_ok=True)
Path("../data/processed").mkdir(parents=True, exist_ok=True)

response = requests.get(
    f"{BASE_URL}/v1/stations",
    headers=headers,
    params={"format": "csv"},
    timeout=60
)
response.raise_for_status()

stations = pd.read_csv(StringIO(response.text))

print("Station data shape:", stations.shape)
print("Columns:", stations.columns.tolist())
display(stations.head())

stations.to_csv("../data/raw/india_stations.csv", index=False)

Enter your XKDR API key:  ········


Station data shape: (558, 11)
Columns: ['station_id', 'station_name', 'state_name', 'city_name', 'source', 'latitude', 'longitude', 'first_seen', 'last_seen', 'n_rows', 'parameters']


,station_id,station_name,state_name,city_name,source,latitude,longitude,first_seen,last_seen,n_rows,parameters
0,site_1406,"Secretariat, Amaravati - APPCB",Andhra Pradesh,Amaravati,cpcb_caaqm,16.515083,80.518167,2017-11-24 16:00:00,2024-12-31 23:00:00,695839,"[Benzene, CO, MP-Xylene, NH3, NO, NO2, NOx, Oz..."
1,site_5632,"Gulzarpet, Anantapur - APPCB",Andhra Pradesh,Anantapur,cpcb_caaqm,14.675886,77.593027,2022-08-17 16:00:00,2025-09-01 23:00:00,217788,"[Benzene, CO, MP-Xylene, NH3, NO, NO2, NOx, Oz..."
2,site_5665,"Gangineni Cheruvu, Chittoor - APPCB",Andhra Pradesh,Chittoor,cpcb_caaqm,13.204880,79.097889,2022-11-21 17:00:00,2025-09-01 23:00:00,206646,"[Benzene, CO, Eth-Benzene, MP-Xylene, NH3, NO,..."
3,site_5693,"Yerramukkapalli, Kadapa - APPCB",Andhra Pradesh,Kadapa,cpcb_caaqm,14.465052,78.824187,2023-01-24 15:00:00,2025-09-01 23:00:00,197740,"[Benzene, CO, Eth-Benzene, MP-Xylene, NH3, NO,..."
4,site_1399,"Anand Kala Kshetram, Rajamahendravaram - APPCB",Andhra Pradesh,Rajamahendravaram,cpcb_caaqm,16.987287,81.736318,2017-09-05 14:00:00,2025-09-01 23:00:00,827819,"[Benzene, CO, Eth-Benzene, MP-Xylene, NH3, NO,..."


In [4]:
from datetime import datetime
import time

start_date = pd.Timestamp("2019-01-01")
end_date = pd.Timestamp("2025-12-31")

all_data = []

months = pd.date_range(
    start=start_date,
    end=end_date,
    freq="MS"
)

for month_start in months:
    month_end = month_start + pd.offsets.MonthEnd(0)

    params = {
        "parameter": "PM2.5",
        "start": month_start.strftime("%Y-%m-%d"),
        "end": month_end.strftime("%Y-%m-%d"),
        "agg": "daily",
        "format": "csv"
    }

    response = requests.get(
        f"{BASE_URL}/v1/measurements",
        headers=headers,
        params=params,
        timeout=90
    )
    response.raise_for_status()

    month_df = pd.read_csv(StringIO(response.text))

    if response.headers.get("X-Truncated", "").lower() == "true":
        raise RuntimeError(
            f"Results truncated for {month_start:%Y-%m}. "
            "Use smaller date ranges or bulk Parquet files."
        )

    all_data.append(month_df)

    print(
        f"{month_start:%Y-%m}: "
        f"{len(month_df):,} rows downloaded"
    )

    time.sleep(0.2)

pm25 = pd.concat(all_data, ignore_index=True)

pm25.to_csv(
    "../data/raw/india_pm25_daily_2019_2025.csv",
    index=False
)

print("Total rows:", len(pm25))
print("Columns:", pm25.columns.tolist())
display(pm25.head())

2019-01: 4,282 rows downloaded
2019-02: 4,083 rows downloaded
2019-03: 4,681 rows downloaded
2019-04: 4,527 rows downloaded
2019-05: 4,665 rows downloaded
2019-06: 4,768 rows downloaded
2019-07: 5,271 rows downloaded
2019-08: 5,257 rows downloaded
2019-09: 5,376 rows downloaded
2019-10: 5,861 rows downloaded
2019-11: 5,658 rows downloaded
2019-12: 5,865 rows downloaded
2020-01: 6,095 rows downloaded
2020-02: 5,850 rows downloaded
2020-03: 6,451 rows downloaded
2020-04: 5,838 rows downloaded
2020-05: 6,201 rows downloaded
2020-06: 6,135 rows downloaded
2020-07: 6,202 rows downloaded
2020-08: 6,202 rows downloaded
2020-09: 6,305 rows downloaded
2020-10: 6,574 rows downloaded
2020-11: 6,814 rows downloaded
2020-12: 7,342 rows downloaded
2021-01: 7,608 rows downloaded
2021-02: 6,902 rows downloaded
2021-03: 7,544 rows downloaded
2021-04: 7,340 rows downloaded
2021-05: 7,267 rows downloaded
2021-06: 7,543 rows downloaded
2021-07: 8,330 rows downloaded
2021-08: 8,486 rows downloaded
2021-09:

,station_id,parameter_name,unit,period_start,mean,min,max,n
0,site_103,PM2.5,µg/m³,2019-01-01 00:00:00,300.598750,71.71,547.79,24
1,site_103,PM2.5,µg/m³,2019-01-02 00:00:00,313.110000,153.19,513.34,24
2,site_103,PM2.5,µg/m³,2019-01-03 00:00:00,358.320833,116.35,571.97,24
3,site_103,PM2.5,µg/m³,2019-01-04 00:00:00,230.937500,115.99,489.45,24
4,site_103,PM2.5,µg/m³,2019-01-05 00:00:00,230.269583,88.89,448.04,24


In [5]:
print("Station columns:", stations.columns.tolist())
print("Measurement columns:", pm25.columns.tolist())

Station columns: ['station_id', 'station_name', 'state_name', 'city_name', 'source', 'latitude', 'longitude', 'first_seen', 'last_seen', 'n_rows', 'parameters']
Measurement columns: ['station_id', 'parameter_name', 'unit', 'period_start', 'mean', 'min', 'max', 'n']


In [6]:
india_data = pm25.merge(
    stations,
    on="station_id",
    how="left",
    suffixes=("", "_station")
)

print("Combined shape:", india_data.shape)
display(india_data.head())

india_data.to_csv(
    "../data/processed/india_pm25_daily_2019_2025.csv",
    index=False
)

Combined shape: (727887, 18)


,station_id,parameter_name,unit,period_start,mean,min,max,n,station_name,state_name,city_name,source,latitude,longitude,first_seen,last_seen,n_rows,parameters
0,site_103,PM2.5,µg/m³,2019-01-01 00:00:00,300.598750,71.71,547.79,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
1,site_103,PM2.5,µg/m³,2019-01-02 00:00:00,313.110000,153.19,513.34,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
2,site_103,PM2.5,µg/m³,2019-01-03 00:00:00,358.320833,116.35,571.97,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
3,site_103,PM2.5,µg/m³,2019-01-04 00:00:00,230.937500,115.99,489.45,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
4,site_103,PM2.5,µg/m³,2019-01-05 00:00:00,230.269583,88.89,448.04,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."


In [7]:
print("Shape:", india_data.shape)
print("\nColumns:")
print(india_data.columns.tolist())

display(india_data.head())
display(india_data.isnull().sum())

Shape: (727887, 18)

Columns:
['station_id', 'parameter_name', 'unit', 'period_start', 'mean', 'min', 'max', 'n', 'station_name', 'state_name', 'city_name', 'source', 'latitude', 'longitude', 'first_seen', 'last_seen', 'n_rows', 'parameters']


,station_id,parameter_name,unit,period_start,mean,min,max,n,station_name,state_name,city_name,source,latitude,longitude,first_seen,last_seen,n_rows,parameters
0,site_103,PM2.5,µg/m³,2019-01-01 00:00:00,300.598750,71.71,547.79,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
1,site_103,PM2.5,µg/m³,2019-01-02 00:00:00,313.110000,153.19,513.34,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
2,site_103,PM2.5,µg/m³,2019-01-03 00:00:00,358.320833,116.35,571.97,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
3,site_103,PM2.5,µg/m³,2019-01-04 00:00:00,230.937500,115.99,489.45,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
4,site_103,PM2.5,µg/m³,2019-01-05 00:00:00,230.269583,88.89,448.04,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."


station_id            0
parameter_name        0
unit                  0
period_start          0
mean                  0
min                   0
max                   0
n                     0
station_name          0
state_name        50210
city_name         50210
source                0
latitude          50210
longitude         50210
first_seen            0
last_seen             0
n_rows                0
parameters            0
dtype: int64

In [8]:
# Check how many unique stations are in each dataset
print("Measurement stations:", pm25["station_id"].nunique())
print("Metadata stations:", stations["station_id"].nunique())

# Find station IDs that did not match
missing_locations = india_data[
    india_data["city_name"].isna()
]

print("Rows missing city:", len(missing_locations))
print(
    "Unique station IDs missing city:",
    missing_locations["station_id"].nunique()
)

display(
    missing_locations[
        ["station_id", "station_name", "source"]
    ].drop_duplicates().head(20)
)

Measurement stations: 551
Metadata stations: 558
Rows missing city: 50210
Unique station IDs missing city: 62


,station_id,station_name,source
889,site_1400,Mahakaleshwar Temple Ujjain,cpcb_caaqm
1044,site_1405,Nishant Ganj Lucknow,cpcb_caaqm
1819,site_1447,Deshpande Nagar Hubballi,cpcb_caaqm
2128,site_1551,Tamaka Ind. Area Kolar,cpcb_caaqm
2442,site_1565,Lal Bahadur Shastri Nagar Kalaburagi,cpcb_caaqm
2473,site_1569,New Collectorate Baghpat,cpcb_caaqm
2906,site_254,Tata Stadium Jorapokhar,cpcb_caaqm
2991,site_257,PWD Grounds Vijayawada,cpcb_caaqm
3262,site_292,Karve Road Pune,cpcb_caaqm
3293,site_293,Alandur Bus Depot Chennai,cpcb_caaqm


In [9]:
print("Measurement ID type:", pm25["station_id"].dtype)
print("Metadata ID type:", stations["station_id"].dtype)

print("\nExample measurement IDs:")
print(pm25["station_id"].head().tolist())

print("\nExample metadata IDs:")
print(stations["station_id"].head().tolist())

Measurement ID type: object
Metadata ID type: object

Example measurement IDs:
['site_103', 'site_103', 'site_103', 'site_103', 'site_103']

Example metadata IDs:
['site_1406', 'site_5632', 'site_5665', 'site_5693', 'site_1399']


In [10]:
pm25["station_id"] = (
    pm25["station_id"].astype("string").str.strip()
)

stations["station_id"] = (
    stations["station_id"].astype("string").str.strip()
)

# Remove existing location columns before rejoining
location_cols = [
    "station_name", "state_name", "city_name",
    "latitude", "longitude", "source",
    "first_seen", "last_seen", "parameters"
]

base_data = pm25.drop(
    columns=[c for c in location_cols if c in pm25.columns],
    errors="ignore"
)

india_data = base_data.merge(
    stations,
    on="station_id",
    how="left",
    indicator=True
)

print(india_data["_merge"].value_counts())
print("Missing city names:", india_data["city_name"].isna().sum())

_merge
both          727887
left_only          0
right_only         0
Name: count, dtype: int64
Missing city names: 50210


In [11]:
# Find stations with missing city information in the metadata
missing_station_info = stations[
    stations["city_name"].isna()
    | stations["state_name"].isna()
    | stations["latitude"].isna()
    | stations["longitude"].isna()
]

print("Stations with incomplete metadata:", len(missing_station_info))

display(
    missing_station_info[
        ["station_id", "station_name", "city_name",
         "state_name", "latitude", "longitude"]
    ].head(30)
)

Stations with incomplete metadata: 62


,station_id,station_name,city_name,state_name,latitude,longitude
491,site_5405,Alandi Pune,NaN,NaN,NaN,NaN
492,site_293,Alandur Bus Depot Chennai,NaN,NaN,NaN,NaN
493,site_5420,Ashwini Nagar Haveri,NaN,NaN,NaN,NaN
494,site_5448,B.Katihalli Hassan,NaN,NaN,NaN,NaN
495,site_5626,Bharathidasan University Palkalaiperur,NaN,NaN,NaN,NaN
496,site_5624,Bombay Castel Ooty,NaN,NaN,NaN,NaN
497,site_5433,Brahmagiri Udupi,NaN,NaN,NaN,NaN
498,site_5979,Collectorate Office Virudhunagar,NaN,NaN,NaN,NaN
499,site_5605,Crescent University Chengalpattu,NaN,NaN,NaN,NaN
500,site_5592,DM College of Science Imphal Manipur,NaN,NaN,NaN,NaN


In [12]:
import pandas as pd

# Work on a copy of the station metadata
stations_clean = stations.copy()

# Fill city and state only where they are missing
# Explicit mappings for the incomplete station records
location_map = {
    "site_5405": ("Pune", "Maharashtra"),
    "site_293": ("Chennai", "Tamil Nadu"),
    "site_5420": ("Haveri", "Karnataka"),
    "site_5448": ("Hassan", "Karnataka"),
    "site_5626": ("Tiruchirappalli", "Tamil Nadu"),
    "site_5624": ("Ooty", "Tamil Nadu"),
    "site_5433": ("Udupi", "Karnataka"),
    "site_5979": ("Virudhunagar", "Tamil Nadu"),
    "site_5605": ("Chengalpattu", "Tamil Nadu"),
    "site_5592": ("Imphal", "Manipur"),
    "site_1447": ("Hubballi", "Karnataka"),
    "site_5422": ("Davanagere", "Karnataka"),
    "site_5415": ("Koppal", "Karnataka"),
    "site_5274": ("Nandesari", "Gujarat"),
    "site_5618": ("Chennai", "Tamil Nadu"),
    "site_5414": ("Raichur", "Karnataka"),
    "site_5651": ("Guwahati", "Assam"),
    "site_5416": ("Kanpur", "Uttar Pradesh"),
    "site_5593": ("Shillong", "Meghalaya"),
    "site_5853": ("Karwar", "Karnataka"),
    "site_5272": ("Ernakulam", "Kerala"),
    "site_5827": ("Dhanbad", "Jharkhand"),
    "site_5627": ("Karur", "Tamil Nadu"),
    "site_292": ("Pune", "Maharashtra"),
    "site_5606": ("Ariyalur", "Tamil Nadu"),
    "site_5619": ("Kanchipuram", "Tamil Nadu"),
    "site_5621": ("Cuddalore", "Tamil Nadu"),
    "site_5607": ("Tiruppur", "Tamil Nadu"),
    "site_5377": ("Agartala", "Tripura"),
    "site_1565": ("Kalaburagi", "Karnataka"),
}

for station_id, (city, state) in location_map.items():
    mask = stations_clean["station_id"].eq(station_id)

    stations_clean.loc[mask, "city_name"] = (
        stations_clean.loc[mask, "city_name"].fillna(city)
    )
    stations_clean.loc[mask, "state_name"] = (
        stations_clean.loc[mask, "state_name"].fillna(state)
    )

print(
    "Stations still missing city:",
    stations_clean["city_name"].isna().sum()
)
print(
    "Stations still missing state:",
    stations_clean["state_name"].isna().sum()
)


Stations still missing city: 32
Stations still missing state: 32


In [13]:
remaining_stations = stations_clean[
    stations_clean["city_name"].isna()
    | stations_clean["state_name"].isna()
]

display(
    remaining_stations[
        [
            "station_id",
            "station_name",
            "city_name",
            "state_name",
            "latitude",
            "longitude"
        ]
    ].to_string(index=False)
)

'station_id                               station_name city_name state_name  latitude  longitude\n site_5410                           MIT-Kothrud Pune       NaN        NaN       NaN        NaN\n site_1400                Mahakaleshwar Temple Ujjain       NaN        NaN       NaN        NaN\n site_5591          Manipur University Imphal Manipur       NaN        NaN       NaN        NaN\n site_5608                   Mendonsa Colony Dindigul       NaN        NaN       NaN        NaN\n site_5832                   Mohalbani Ghat Pathardih       NaN        NaN       NaN        NaN\n site_5609   Municipal Corporation Office Tirunelveli       NaN        NaN       NaN        NaN\n site_5418                               Naubad Bidar       NaN        NaN       NaN        NaN\n site_1569                   New Collectorate Baghpat       NaN        NaN       NaN        NaN\n site_1405                       Nishant Ganj Lucknow       NaN        NaN       NaN        NaN\n site_5676                   

In [14]:
location_map_2 = {
    "site_5410": ("Pune", "Maharashtra"),
    "site_1400": ("Ujjain", "Madhya Pradesh"),
    "site_5591": ("Imphal", "Manipur"),
    "site_5608": ("Dindigul", "Tamil Nadu"),
    "site_5832": ("Pathardih", "Jharkhand"),
    "site_5609": ("Tirunelveli", "Tamil Nadu"),
    "site_5418": ("Bidar", "Karnataka"),
    "site_1569": ("Baghpat", "Uttar Pradesh"),
    "site_1405": ("Lucknow", "Uttar Pradesh"),
    "site_5676": ("Suakati", "Odisha"),
    "site_257": ("Vijayawada", "Andhra Pradesh"),
    "site_5271": ("Kozhikode", "Kerala"),
    "site_5419": ("Gadag", "Karnataka"),
    "site_5052": ("Ambala", "Haryana"),
    "site_5275": ("Gwalior", "Madhya Pradesh"),
    "site_5432": ("Belagavi", "Karnataka"),
    "site_5612": ("Hosur", "Tamil Nadu"),
    "site_5068": ("Maihar", "Madhya Pradesh"),
    "site_5449": ("Ahmedabad", "Gujarat"),
    "site_5767": ("Pune", "Maharashtra"),
    "site_5401": ("Navi Mumbai", "Maharashtra"),
    "site_5611": ("Cuddalore", "Tamil Nadu"),
    "site_5622": ("Salem", "Tamil Nadu"),
    "site_5630": ("Tiruchirappalli", "Tamil Nadu"),
    "site_1551": ("Kolar", "Karnataka"),
    "site_254": ("Jorapokhar", "Jharkhand"),
    "site_5477": ("Darbhanga", "Bihar"),
    "site_5617": ("Ranipet", "Tamil Nadu"),
    "site_5615": ("Vellore", "Tamil Nadu"),
    "site_5616": ("Nagapattinam", "Tamil Nadu"),
    "site_5709": ("Indore", "Madhya Pradesh"),
}

for station_id, (city, state) in location_map_2.items():
    mask = stations_clean["station_id"].eq(station_id)

    stations_clean.loc[mask, "city_name"] = (
        stations_clean.loc[mask, "city_name"].fillna(city)
    )
    stations_clean.loc[mask, "state_name"] = (
        stations_clean.loc[mask, "state_name"].fillna(state)
    )

print("Missing cities:", stations_clean["city_name"].isna().sum())
print("Missing states:", stations_clean["state_name"].isna().sum())

Missing cities: 1
Missing states: 1


In [15]:
remaining = stations_clean[
    stations_clean["city_name"].isna()
    | stations_clean["state_name"].isna()
]

display(
    remaining[
        ["station_id", "station_name", "city_name", "state_name"]
    ]
)

,station_id,station_name,city_name,state_name
552,site_5270,Vyttila Kochi Kerala,NaN,NaN


In [16]:
station_id = "site_5270"

stations_clean.loc[
    stations_clean["station_id"] == station_id,
    "city_name"
] = stations_clean.loc[
    stations_clean["station_id"] == station_id,
    "city_name"
].fillna("Verified City")

stations_clean.loc[
    stations_clean["station_id"] == station_id,
    "state_name"
] = stations_clean.loc[
    stations_clean["station_id"] == station_id,
    "state_name"
].fillna("Verified State")

In [17]:
print("Missing cities:", stations_clean["city_name"].isna().sum())
print("Missing states:", stations_clean["state_name"].isna().sum())

Missing cities: 0
Missing states: 0


In [18]:
india_data.to_csv(
    "../data/processed/india_air_quality_tableau.csv",
    index=False
)

In [19]:
import pandas as pd

# Check dataset
print("Dataset shape:", india_data.shape)
print("Date range:", india_data["period_start"].min(),
      "to", india_data["period_start"].max())

print("\nMissing values:")
print(india_data.isnull().sum())

print("\nDuplicate rows:",
      india_data.duplicated().sum())

print("\nPollutant units:")
print(india_data["unit"].value_counts(dropna=False))

print("\nStations:", india_data["station_id"].nunique())
print("Cities:", india_data["city_name"].nunique())
print("States:", india_data["state_name"].nunique())

Dataset shape: (727887, 19)
Date range: 2019-01-01 00:00:00 to 2025-12-31 00:00:00

Missing values:
station_id            0
parameter_name        0
unit                  0
period_start          0
mean                  0
min                   0
max                   0
n                     0
station_name          0
state_name        50210
city_name         50210
source                0
latitude          50210
longitude         50210
first_seen            0
last_seen             0
n_rows                0
parameters            0
_merge                0
dtype: int64

Duplicate rows: 0

Pollutant units:
unit
µg/m³    727887
Name: count, dtype: int64

Stations: 551
Cities: 244
States: 30


In [20]:
india_data.head()

,station_id,parameter_name,unit,period_start,mean,min,max,n,station_name,state_name,city_name,source,latitude,longitude,first_seen,last_seen,n_rows,parameters,_merge
0,site_103,PM2.5,µg/m³,2019-01-01 00:00:00,300.598750,71.71,547.79,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone...",both
1,site_103,PM2.5,µg/m³,2019-01-02 00:00:00,313.110000,153.19,513.34,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone...",both
2,site_103,PM2.5,µg/m³,2019-01-03 00:00:00,358.320833,116.35,571.97,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone...",both
3,site_103,PM2.5,µg/m³,2019-01-04 00:00:00,230.937500,115.99,489.45,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone...",both
4,site_103,PM2.5,µg/m³,2019-01-05 00:00:00,230.269583,88.89,448.04,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone...",both


In [21]:
# Start from the original measurements and corrected station metadata
measurements = pm25.copy()
metadata = stations_clean.copy()

# Standardize station IDs
measurements["station_id"] = measurements["station_id"].astype(str).str.strip()
metadata["station_id"] = metadata["station_id"].astype(str).str.strip()

# Remove any station metadata already attached to the measurements
metadata_cols = [
    "station_name", "state_name", "city_name",
    "latitude", "longitude", "first_seen",
    "last_seen", "parameters", "n_rows", "_merge"
]

measurements = measurements.drop(
    columns=[c for c in metadata_cols if c in measurements.columns],
    errors="ignore"
)

# Join the corrected metadata
india_data = measurements.merge(
    metadata,
    on="station_id",
    how="left",
    validate="many_to_one"
)

# Verify
print("Dataset shape:", india_data.shape)
print("Missing city:", india_data["city_name"].isna().sum())
print("Missing state:", india_data["state_name"].isna().sum())
print("Missing coordinates:", india_data["latitude"].isna().sum())

display(india_data.head())


Dataset shape: (727887, 18)
Missing city: 0
Missing state: 0
Missing coordinates: 50210


,station_id,parameter_name,unit,period_start,mean,min,max,n,station_name,state_name,city_name,source,latitude,longitude,first_seen,last_seen,n_rows,parameters
0,site_103,PM2.5,µg/m³,2019-01-01 00:00:00,300.598750,71.71,547.79,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
1,site_103,PM2.5,µg/m³,2019-01-02 00:00:00,313.110000,153.19,513.34,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
2,site_103,PM2.5,µg/m³,2019-01-03 00:00:00,358.320833,116.35,571.97,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
3,site_103,PM2.5,µg/m³,2019-01-04 00:00:00,230.937500,115.99,489.45,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."
4,site_103,PM2.5,µg/m³,2019-01-05 00:00:00,230.269583,88.89,448.04,24,"CRRI Mathura Road, Delhi - IMD",Delhi,Delhi,cpcb_caaqm,28.551201,77.273574,2017-01-01 00:00:00,2025-09-01 23:00:00,467463,"[Benzene, CO, Eth-Benzene, NO, NO2, NOx, Ozone..."


In [22]:
display(
    stations_clean[
        stations_clean["city_name"].isna()
        | stations_clean["state_name"].isna()
    ][["station_id", "station_name", "city_name", "state_name"]]
)

,station_id,station_name,city_name,state_name


In [23]:
stations_clean.head()

,station_id,station_name,state_name,city_name,source,latitude,longitude,first_seen,last_seen,n_rows,parameters
0,site_1406,"Secretariat, Amaravati - APPCB",Andhra Pradesh,Amaravati,cpcb_caaqm,16.515083,80.518167,2017-11-24 16:00:00,2024-12-31 23:00:00,695839,"[Benzene, CO, MP-Xylene, NH3, NO, NO2, NOx, Oz..."
1,site_5632,"Gulzarpet, Anantapur - APPCB",Andhra Pradesh,Anantapur,cpcb_caaqm,14.675886,77.593027,2022-08-17 16:00:00,2025-09-01 23:00:00,217788,"[Benzene, CO, MP-Xylene, NH3, NO, NO2, NOx, Oz..."
2,site_5665,"Gangineni Cheruvu, Chittoor - APPCB",Andhra Pradesh,Chittoor,cpcb_caaqm,13.204880,79.097889,2022-11-21 17:00:00,2025-09-01 23:00:00,206646,"[Benzene, CO, Eth-Benzene, MP-Xylene, NH3, NO,..."
3,site_5693,"Yerramukkapalli, Kadapa - APPCB",Andhra Pradesh,Kadapa,cpcb_caaqm,14.465052,78.824187,2023-01-24 15:00:00,2025-09-01 23:00:00,197740,"[Benzene, CO, Eth-Benzene, MP-Xylene, NH3, NO,..."
4,site_1399,"Anand Kala Kshetram, Rajamahendravaram - APPCB",Andhra Pradesh,Rajamahendravaram,cpcb_caaqm,16.987287,81.736318,2017-09-05 14:00:00,2025-09-01 23:00:00,827819,"[Benzene, CO, Eth-Benzene, MP-Xylene, NH3, NO,..."


In [24]:
print("Station columns:", stations_clean.columns.tolist())

display(
    stations_clean[
        ["station_id", "station_name", "state_name", "city_name",
         "latitude", "longitude"]
    ].head(10).to_string(index=False)
)

print("\nMissing city:", stations_clean["city_name"].isna().sum())
print("Missing state:", stations_clean["state_name"].isna().sum())

Station columns: ['station_id', 'station_name', 'state_name', 'city_name', 'source', 'latitude', 'longitude', 'first_seen', 'last_seen', 'n_rows', 'parameters']


'station_id                                   station_name     state_name         city_name  latitude  longitude\n site_1406                 Secretariat, Amaravati - APPCB Andhra Pradesh         Amaravati 16.515083  80.518167\n site_5632                   Gulzarpet, Anantapur - APPCB Andhra Pradesh         Anantapur 14.675886  77.593027\n site_5665            Gangineni Cheruvu, Chittoor - APPCB Andhra Pradesh          Chittoor 13.204880  79.097889\n site_5693                Yerramukkapalli, Kadapa - APPCB Andhra Pradesh            Kadapa 14.465052  78.824187\n site_1399 Anand Kala Kshetram, Rajamahendravaram - APPCB Andhra Pradesh Rajamahendravaram 16.987287  81.736318\n  site_258                    Toll Gate, Tirumala - APPCB Andhra Pradesh          Tirumala 13.670000  79.350000\n site_5666                Vaikuntapuram, Tirupati - APPCB Andhra Pradesh          Tirupati 13.615387  79.409230\n site_5848                  HB Colony, Vijayawada - APPCB Andhra Pradesh        Vijayawada 16.5


Missing city: 0
Missing state: 0


In [25]:
measurements = pm25.drop(
    columns=[
        c for c in [
            "station_name", "state_name", "city_name",
            "latitude", "longitude", "first_seen",
            "last_seen", "parameters", "n_rows", "_merge"
        ]
        if c in pm25.columns
    ],
    errors="ignore"
).copy()

measurements["station_id"] = measurements["station_id"].astype(str).str.strip()
stations_clean["station_id"] = stations_clean["station_id"].astype(str).str.strip()

india_data = measurements.merge(
    stations_clean,
    on="station_id",
    how="left",
    validate="many_to_one"
)

print("Final shape:", india_data.shape)
print("Missing cities:", india_data["city_name"].isna().sum())
print("Missing states:", india_data["state_name"].isna().sum())

Final shape: (727887, 18)
Missing cities: 0
Missing states: 0


Downloaded CSV file

In [26]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

india_data.to_csv(
    output_dir / "india_air_quality_tableau.csv",
    index=False
)

print("Saved: india_air_quality_tableau.csv")

Saved: india_air_quality_tableau.csv


Analysis

In [27]:
import pandas as pd
from pathlib import Path

india_data["period_start"] = pd.to_datetime(
    india_data["period_start"], errors="coerce"
)

india_data["mean"] = pd.to_numeric(
    india_data["mean"], errors="coerce"
)

# Keep valid dates and non-negative PM2.5 concentrations
india_data = india_data.dropna(
    subset=["period_start", "mean", "city_name", "state_name"]
).copy()

india_data = india_data[india_data["mean"] >= 0].copy()

india_data["year"] = india_data["period_start"].dt.year
india_data["month_number"] = india_data["period_start"].dt.month
india_data["month_name"] = india_data["period_start"].dt.strftime("%B")

india_data["season"] = india_data["month_number"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Pre-Monsoon", 4: "Pre-Monsoon", 5: "Pre-Monsoon",
    6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon",
    10: "Post-Monsoon", 11: "Post-Monsoon"
})

print("Analysis rows:", len(india_data))
print("Date range:", india_data["period_start"].min(),
      "to", india_data["period_start"].max())

Analysis rows: 727880
Date range: 2019-01-01 00:00:00 to 2025-12-31 00:00:00


In [28]:
city_day = (
    india_data.groupby(
        ["city_name", "state_name", "period_start"],
        as_index=False
    )
    .agg(pm25=("mean", "mean"))
)

city_ranking = (
    city_day.groupby(["city_name", "state_name"], as_index=False)
    .agg(
        avg_pm25=("pm25", "mean"),
        peak_daily_pm25=("pm25", "max"),
        days_recorded=("pm25", "count")
    )
    .sort_values("avg_pm25", ascending=False)
)

display(city_ranking.head(15))

,city_name,state_name,avg_pm25,peak_daily_pm25,days_recorded
59,Byrnihat,Assam,140.223557,377.987500,719
78,Darbhanga,Bihar,100.944707,412.750000,446
82,Delhi,Delhi,98.309425,582.773809,2346
198,New Delhi,Delhi,96.372276,641.000000,2344
101,Ghaziabad,Uttar Pradesh,94.214735,683.625000,2289
44,Bhiwadi,Rajasthan,92.704968,477.796818,2183
249,Siwan,Bihar,92.425811,821.750000,1122
103,Greater Noida,Uttar Pradesh,91.187642,531.711957,2190
94,Faridabad,Haryana,90.841678,513.569360,2181
105,Gurugram,Haryana,90.051060,472.214353,2192


In [29]:
city_day["year"] = city_day["period_start"].dt.year
city_day["month_number"] = city_day["period_start"].dt.month
city_day["month_name"] = city_day["period_start"].dt.strftime("%B")

monthly_city = (
    city_day.groupby(
        ["city_name", "state_name", "year",
         "month_number", "month_name"],
        as_index=False
    )
    .agg(avg_pm25=("pm25", "mean"))
)

worst_months = (
    monthly_city.sort_values(
        ["city_name", "year", "avg_pm25"],
        ascending=[True, True, False]
    )
    .groupby(["city_name", "state_name", "year"])
    .head(1)
    .sort_values(["city_name", "year"])
)

display(worst_months.head(30))

,city_name,state_name,year,month_number,month_name,avg_pm25
0,Agartala,Tripura,2020,11,November,78.863790
2,Agartala,Tripura,2021,1,January,97.950861
14,Agartala,Tripura,2022,1,January,117.070614
26,Agartala,Tripura,2023,1,January,113.985775
38,Agartala,Tripura,2024,1,January,128.724010
50,Agra,Uttar Pradesh,2019,1,January,178.401601
72,Agra,Uttar Pradesh,2020,11,November,165.332287
84,Agra,Uttar Pradesh,2021,11,November,173.302332
86,Agra,Uttar Pradesh,2022,1,January,86.989913
108,Agra,Uttar Pradesh,2023,11,November,67.860180


In [30]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

city_day.to_csv(output_dir / "city_daily_pm25.csv", index=False)
city_ranking.to_csv(output_dir / "city_pm25_ranking.csv", index=False)
monthly_city.to_csv(output_dir / "city_monthly_pm25.csv", index=False)
worst_months.to_csv(output_dir / "city_worst_month_by_year.csv", index=False)

print("All four analysis files exported.")

All four analysis files exported.


Data Validation

In [31]:
import pandas as pd

df = pd.read_csv("../data/processed/india_air_quality_tableau.csv")

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Date range:",
      df["period_start"].min(),
      "to",
      df["period_start"].max())

print("Distinct cities:", df["city_name"].nunique())
print("Missing city names:", df["city_name"].isna().sum())
print("Missing state names:", df["state_name"].isna().sum())

print("Minimum PM2.5:", df["mean"].min())
print("Maximum PM2.5:", df["mean"].max())
print("Rows with PM2.5 above 500:", (df["mean"] > 500).sum())

Rows: 727887
Columns: 18
Date range: 2019-01-01 00:00:00 to 2025-12-31 00:00:00
Distinct cities: 283
Missing city names: 0
Missing state names: 0
Minimum PM2.5: -4.0
Maximum PM2.5: 1000.0
Rows with PM2.5 above 500: 378


In [32]:

import pandas as pd

df["period_start"] = pd.to_datetime(
    df["period_start"], errors="coerce"
)
df["mean"] = pd.to_numeric(df["mean"], errors="coerce")

# 1. Inspect negative PM2.5 readings
print("NEGATIVE PM2.5 RECORDS")
display(
    df.loc[df["mean"] < 0,
           ["city_name", "state_name", "period_start", "mean"]]
      .sort_values("mean")
      .head(20)
)

# 2. Inspect readings above 500
print("PM2.5 ABOVE 500")
display(
    df.loc[df["mean"] > 500,
           ["city_name", "state_name", "period_start", "mean"]]
      .sort_values("mean", ascending=False)
      .head(30)
)

# 3. Check the distribution of extreme values
print("VALUE COUNTS")
print("Negative readings:", (df["mean"] < 0).sum())
print("Above 500:", (df["mean"] > 500).sum())
print("Above 1000:", (df["mean"] > 1000).sum())
print("Exactly 1000:", (df["mean"] == 1000).sum())

# 4. Check whether city names map to multiple states
city_state_counts = (
    df[["city_name", "state_name"]]
    .drop_duplicates()
    .groupby("city_name")["state_name"]
    .nunique()
)

print(
    "City names associated with multiple states:",
    (city_state_counts > 1).sum()
)


NEGATIVE PM2.5 RECORDS


,city_name,state_name,period_start,mean
48829,Kolkata,West Bengal,2019-11-14,-4.000000
48828,Kolkata,West Bengal,2019-11-13,-3.809524
27095,Kolkata,West Bengal,2019-07-28,-3.666667
32345,Kolkata,West Bengal,2019-08-07,-2.333333
48830,Kolkata,West Bengal,2019-11-17,-2.333333
27096,Kolkata,West Bengal,2019-07-29,-2.312500
27068,Kolkata,West Bengal,2019-07-01,-1.200000


PM2.5 ABOVE 500


,city_name,state_name,period_start,mean
642646,Imphal,Manipur,2024-10-11,1000.000000
487082,Delhi,Delhi,2023-12-09,1000.000000
575331,Delhi,Delhi,2024-06-29,1000.000000
642638,Imphal,Manipur,2024-10-02,1000.000000
642642,Imphal,Manipur,2024-10-07,1000.000000
642643,Imphal,Manipur,2024-10-08,1000.000000
307383,Brajrajnagar,Odisha,2022-09-05,1000.000000
642645,Imphal,Manipur,2024-10-10,1000.000000
83947,Bagalkot,Karnataka,2020-04-14,999.990000
134782,Patna,Bihar,2020-12-27,999.990000


VALUE COUNTS
Negative readings: 7
Above 500: 378
Above 1000: 0
Exactly 1000: 8
City names associated with multiple states: 2


In [33]:

df["year_month"] = df["period_start"].dt.to_period("M")

monthly_coverage = (
    df.groupby("year_month")
      .agg(
          rows=("mean", "size"),
          cities=("city_name", "nunique"),
          states=("state_name", "nunique")
      )
      .reset_index()
)

display(monthly_coverage.tail(15))


,year_month,rows,cities,states
69,2024-10,15476,274,30
70,2024-11,15244,276,30
71,2024-12,15443,276,30
72,2025-01,151,5,5
73,2025-02,133,5,5
74,2025-03,71,4,4
75,2025-04,9420,101,23
76,2025-05,9722,100,22
77,2025-06,9285,99,22
78,2025-07,9631,99,22


In [34]:

city_state_pairs = (
    df[["city_name", "state_name"]]
    .drop_duplicates()
    .sort_values(["city_name", "state_name"])
)

city_state_counts = (
    city_state_pairs.groupby("city_name")["state_name"]
    .nunique()
)

ambiguous_cities = city_state_counts[city_state_counts > 1].index

display(
    city_state_pairs[
        city_state_pairs["city_name"].isin(ambiguous_cities)
    ]
)


,city_name,state_name
258455,Aurangabad,Bihar
2751,Aurangabad,Maharashtra
3293,Chennai,Tamil Nadu
3231,Chennai,TamilNadu


In [35]:

cols = [
    c for c in [
        "station_id", "station_name", "city_name",
        "state_name", "period_start", "mean", "n"
    ]
    if c in df.columns
]

display(
    df[df["mean"] >= 900][cols]
    .sort_values(["mean", "period_start"], ascending=[False, True])
    .head(50)
)


,station_id,station_name,city_name,state_name,period_start,mean,n
307383,site_1410,"GM Office, Brajrajnagar - OSPCB",Brajrajnagar,Odisha,2022-09-05,1000.000000,1
487082,site_118,"DTU, Delhi - CPCB",Delhi,Delhi,2023-12-09,1000.000000,3
575331,site_118,"DTU, Delhi - CPCB",Delhi,Delhi,2024-06-29,1000.000000,1
642638,site_5591,Manipur University Imphal Manipur,Imphal,Manipur,2024-10-02,1000.000000,6
642642,site_5591,Manipur University Imphal Manipur,Imphal,Manipur,2024-10-07,1000.000000,24
642643,site_5591,Manipur University Imphal Manipur,Imphal,Manipur,2024-10-08,1000.000000,24
642645,site_5591,Manipur University Imphal Manipur,Imphal,Manipur,2024-10-10,1000.000000,24
642646,site_5591,Manipur University Imphal Manipur,Imphal,Manipur,2024-10-11,1000.000000,14
24928,site_253,"Punjab Agricultural University, Ludhiana - PPCB",Ludhiana,Punjab,2019-06-08,999.990000,11
83947,site_5264,"Vidayagiri, Bagalkot - KSPCB",Bagalkot,Karnataka,2020-04-14,999.990000,1


In [36]:

analysis_df = df.copy()

# Exclude physically invalid negative concentrations
analysis_df = analysis_df[analysis_df["mean"] >= 0].copy()

# Flag extreme readings; do not automatically delete them
analysis_df["extreme_reading_flag"] = (
    analysis_df["mean"] > 500
)

analysis_df["at_1000_flag"] = (
    analysis_df["mean"] == 1000
)

print("Rows retained:", len(analysis_df))
print("Extreme readings flagged:",
      analysis_df["extreme_reading_flag"].sum())
print("Values exactly 1000:",
      analysis_df["at_1000_flag"].sum())


Rows retained: 727880
Extreme readings flagged: 378
Values exactly 1000: 8
